In [0]:
from pyspark.sql import functions as F

# Tables Loading
df_fact_orders = spark.table("restaurant_project.gold.fact_orders")
df_dim_products = spark.table("restaurant_project.gold.dim_products")
df_dim_branches = spark.table("restaurant_project.gold.dim_branches")

## Surrogate Key Uniqueness (Dimensions Integrity)

In [0]:
# 1.1 Product Key Uniqueness
prd_dupes = df_dim_products.groupBy("product_skey").count().filter("count > 1")
if prd_dupes.count() == 0:
    print("✅ Pass: Product Keys are unique.")
else:
    print(f"❌ Fail: Found {prd_dupes.count()} duplicate Product Keys.")
    prd_dupes.show()

# 1.2 Branch Key Uniqueness
br_dupes = df_dim_branches.groupBy("branch_skey").count().filter("count > 1")
if br_dupes.count() == 0:
    print("✅ Pass: Branch Keys are unique.")
else:
    print(f"❌ Fail: Found {br_dupes.count()} duplicate Branch Keys.")

## Referential Integrity

In [0]:
# 2.1 Fact -> Product link
missing_prds = df_fact_orders.join(df_dim_products, "product_skey", "left_anti").count()
if missing_prds == 0:
    print("✅ Pass: All sales are linked to valid Products.")
else:
    print(f"❌ Fail: Found {missing_prds} orphan sales records (No Product link).")

# 2.2 Fact -> Branch link
missing_brs = df_fact_orders.join(df_dim_branches, "branch_skey", "left_anti").count()
if missing_brs == 0:
    print("✅ Pass: All sales are linked to valid Branches.")
else:
    print(f"❌ Fail: Found {missing_brs} orphan sales records (No Branch link).")

## Date & Time Sanity Check


In [0]:
# 4.1 Future Dates Check
current_date = F.current_date()
future_sales = df_fact_orders.filter(F.col("order_date") > current_date).count()

if future_sales == 0:
    print("✅ Pass: No future sales dates detected.")
else:
    print(f"❌ Fail: Found {future_sales} records with future dates!")

# 4.2 Hour Range Check (0-23)
invalid_hours = df_fact_orders.filter((F.col("hour") < 0) | (F.col("hour") > 23)).count()
if invalid_hours == 0:
    print("✅ Pass: Hour values are in valid range (0-23).")
else:
    print(f"❌ Fail: Found {invalid_hours} records with invalid hour values.")